In [ ]:
# CELL 1 — Persiapan Dataset

from google.colab import drive
from pathlib import Path

# Menghubungkan Google Drive
drive.mount('/content/drive')

# Menentukan lokasi dataset O*NET
PROJECT_DIR = Path('/content/drive/MyDrive/CRS')
ONET_DIR = PROJECT_DIR / 'ONET'
ONET_CSV_DIR = ONET_DIR / 'extracted' / 'db_31_0_csv'

print("Lokasi dataset O*NET:")
print(ONET_CSV_DIR)

print("\nFolder tersedia:", ONET_CSV_DIR.exists())

Mounted at /content/drive
Lokasi dataset O*NET:
/content/drive/MyDrive/CRS/ONET/extracted/db_31_0_csv

Folder tersedia: True


In [ ]:
# CELL 2 — Membaca Dataset O*NET

import pandas as pd

# Membaca dataset
occupation_data = pd.read_csv(
    ONET_CSV_DIR / 'occupation_data.csv'
)

career_interest_types = pd.read_csv(
    ONET_CSV_DIR / 'career_interest_types.csv'
)

abilities = pd.read_csv(
    ONET_CSV_DIR / 'abilities.csv'
)

knowledge = pd.read_csv(
    ONET_CSV_DIR / 'knowledge.csv'
)

essential_skills = pd.read_csv(
    ONET_CSV_DIR / 'essential_skills.csv'
)

transferable_skills = pd.read_csv(
    ONET_CSV_DIR / 'transferable_skills.csv'
)

# Menampilkan ukuran masing-masing dataset
print("Occupation Data       :", occupation_data.shape)
print("Career Interest Types :", career_interest_types.shape)
print("Abilities             :", abilities.shape)
print("Knowledge             :", knowledge.shape)
print("Essential Skills      :", essential_skills.shape)
print("Transferable Skills   :", transferable_skills.shape)

Occupation Data       : (1016, 3)
Career Interest Types : (8307, 9)
Abilities             : (94640, 15)
Knowledge             : (60060, 15)
Essential Skills      : (18200, 15)
Transferable Skills   : (45500, 15)


In [ ]:
# CELL 3 — Membentuk Profil RIASEC

# Enam dimensi RIASEC
riasec = [
    'Realistic',
    'Investigative',
    'Artistic',
    'Social',
    'Enterprising',
    'Conventional'
]

# Mengambil data RIASEC
riasec_data = career_interest_types[
    career_interest_types['Element Name'].isin(riasec)
].copy()

# Mengubah data menjadi bentuk:
# satu baris = satu pekerjaan
# satu kolom = satu dimensi RIASEC
riasec_profile = riasec_data.pivot_table(
    index=['O*NET-SOC Code', 'Title'],
    columns='Element Name',
    values='Data Value'
).reset_index()

# Memastikan urutan kolom RIASEC
riasec_profile = riasec_profile[
    ['O*NET-SOC Code', 'Title'] + riasec
]

print("Jumlah pekerjaan :", len(riasec_profile))
print("Jumlah fitur     :", len(riasec))
print("\nContoh profil RIASEC:")
display(riasec_profile.head())

Jumlah pekerjaan : 923
Jumlah fitur     : 6

Contoh profil RIASEC:


Element Name,O*NET-SOC Code,Title,Realistic,Investigative,Artistic,Social,Enterprising,Conventional
0,11-1011.00,Chief Executives,1.26,3.05,2.16,3.54,6.96,4.97
1,11-1011.03,Chief Sustainability Officers,2.04,4.78,2.48,3.55,6.68,4.49
2,11-1021.00,General and Operations Managers,2.20,2.37,1.29,3.38,7.00,5.34
3,11-1031.00,Legislators,1.54,3.35,2.70,3.69,5.52,3.62
4,11-2011.00,Advertising and Promotions Managers,1.00,1.69,3.87,3.14,7.00,4.43


In [ ]:
# CELL 4 — Membentuk Profil Kompetensi O*NET

# Fungsi untuk mengubah data O*NET menjadi profil kompetensi
def create_competency_profile(data, domain):
    profile = data.pivot_table(
        index=['O*NET-SOC Code', 'Title'],
        columns='Element Name',
        values='Data Value'
    ).reset_index()

    profile['Domain'] = domain

    return profile


# Membentuk profil masing-masing domain
abilities_profile = create_competency_profile(
    abilities,
    'Abilities'
)

knowledge_profile = create_competency_profile(
    knowledge,
    'Knowledge'
)

essential_skills_profile = create_competency_profile(
    essential_skills,
    'Essential Skills'
)

transferable_skills_profile = create_competency_profile(
    transferable_skills,
    'Transferable Skills'
)


print("Abilities          :", abilities_profile.shape)
print("Knowledge          :", knowledge_profile.shape)
print("Essential Skills   :", essential_skills_profile.shape)
print("Transferable Skills:", transferable_skills_profile.shape)

Abilities          : (910, 55)
Knowledge          : (910, 36)
Essential Skills   : (910, 13)
Transferable Skills: (910, 28)


In [ ]:
# CELL 5 — Menggabungkan Profil O*NET

# Mengubah masing-masing domain menjadi bentuk long format
def to_long_format(data, domain):
    feature_columns = [
        column for column in data.columns
        if column not in ['O*NET-SOC Code', 'Title', 'Domain']
    ]

    result = data.melt(
        id_vars=['O*NET-SOC Code', 'Title'],
        value_vars=feature_columns,
        var_name='Feature',
        value_name='Level'
    )

    result['Domain'] = domain

    return result


abilities_long = to_long_format(
    abilities_profile,
    'Abilities'
)

knowledge_long = to_long_format(
    knowledge_profile,
    'Knowledge'
)

essential_skills_long = to_long_format(
    essential_skills_profile,
    'Essential Skills'
)

transferable_skills_long = to_long_format(
    transferable_skills_profile,
    'Transferable Skills'
)


# Menggabungkan seluruh kompetensi
competency_profile = pd.concat(
    [
        abilities_long,
        knowledge_long,
        essential_skills_long,
        transferable_skills_long
    ],
    ignore_index=True
)


print("Jumlah data kompetensi :", len(competency_profile))
print("Jumlah pekerjaan       :", competency_profile['O*NET-SOC Code'].nunique())
print("Jumlah domain          :", competency_profile['Domain'].nunique())

print("\nContoh data:")
display(competency_profile.head())

Jumlah data kompetensi : 109200
Jumlah pekerjaan       : 910
Jumlah domain          : 4

Contoh data:


,O*NET-SOC Code,Title,Feature,Level,Domain
0,11-1011.00,Chief Executives,Arm-Hand Steadiness,0.94,Abilities
1,11-1011.03,Chief Sustainability Officers,Arm-Hand Steadiness,0.50,Abilities
2,11-1021.00,General and Operations Managers,Arm-Hand Steadiness,1.25,Abilities
3,11-2011.00,Advertising and Promotions Managers,Arm-Hand Steadiness,0.88,Abilities
4,11-2021.00,Marketing Managers,Arm-Hand Steadiness,0.88,Abilities


In [ ]:
# CELL 6 — Membuat Profil Siswa

# Profil minat siswa (RIASEC)
student_riasec = {
    'Realistic': 0.65,
    'Investigative': 0.90,
    'Artistic': 0.40,
    'Social': 0.50,
    'Enterprising': 0.35,
    'Conventional': 0.60
}

# Contoh skor kompetensi siswa
student_competency = {
    'Deductive Reasoning': 0.85,
    'Inductive Reasoning': 0.80,
    'Mathematical Reasoning': 0.75,
    'Critical Thinking': 0.90,
    'Complex Problem Solving': 0.85,
    'Systems Analysis': 0.80,
    'Systems Evaluation': 0.75,
    'Programming': 0.85,
    'Technology Design': 0.80,
    'Written Expression': 0.70,
    'Oral Comprehension': 0.75,
    'Active Listening': 0.70,
    'Reading Comprehension': 0.80,
    'Speaking': 0.65,
    'Social Perceptiveness': 0.55,
    'Service Orientation': 0.50
}

# Menampilkan profil siswa
print("Profil RIASEC:")
for feature, score in student_riasec.items():
    print(f"{feature:<15} : {score}")

print("\nProfil Kompetensi:")
for feature, score in student_competency.items():
    print(f"{feature:<25} : {score}")

Profil RIASEC:
Realistic       : 0.65
Investigative   : 0.9
Artistic        : 0.4
Social          : 0.5
Enterprising    : 0.35
Conventional    : 0.6

Profil Kompetensi:
Deductive Reasoning       : 0.85
Inductive Reasoning       : 0.8
Mathematical Reasoning    : 0.75
Critical Thinking         : 0.9
Complex Problem Solving   : 0.85
Systems Analysis          : 0.8
Systems Evaluation        : 0.75
Programming               : 0.85
Technology Design         : 0.8
Written Expression        : 0.7
Oral Comprehension        : 0.75
Active Listening          : 0.7
Reading Comprehension     : 0.8
Speaking                  : 0.65
Social Perceptiveness     : 0.55
Service Orientation       : 0.5


In [ ]:
# CELL 7 — Normalisasi Profil O*NET

# Normalisasi nilai RIASEC O*NET dari skala 1–7 menjadi 0–1
riasec_profile_normalized = riasec_profile.copy()

for feature in riasec:
    riasec_profile_normalized[feature] = (
        riasec_profile_normalized[feature] - 1
    ) / 6


# Normalisasi Level kompetensi O*NET dari skala 0–7 menjadi 0–1
competency_profile_normalized = competency_profile.copy()

competency_profile_normalized['Level_Normalized'] = (
    competency_profile_normalized['Level'] / 7
)


print("Contoh RIASEC setelah normalisasi:")
display(
    riasec_profile_normalized[
        ['O*NET-SOC Code', 'Title'] + riasec
    ].head()
)

print("\nContoh kompetensi setelah normalisasi:")
display(
    competency_profile_normalized.head()
)

Contoh RIASEC setelah normalisasi:


Element Name,O*NET-SOC Code,Title,Realistic,Investigative,Artistic,Social,Enterprising,Conventional
0,11-1011.00,Chief Executives,0.043333,0.341667,0.193333,0.423333,0.993333,0.661667
1,11-1011.03,Chief Sustainability Officers,0.173333,0.630000,0.246667,0.425000,0.946667,0.581667
2,11-1021.00,General and Operations Managers,0.200000,0.228333,0.048333,0.396667,1.000000,0.723333
3,11-1031.00,Legislators,0.090000,0.391667,0.283333,0.448333,0.753333,0.436667
4,11-2011.00,Advertising and Promotions Managers,0.000000,0.115000,0.478333,0.356667,1.000000,0.571667



Contoh kompetensi setelah normalisasi:


,O*NET-SOC Code,Title,Feature,Level,Domain,Level_Normalized
0,11-1011.00,Chief Executives,Arm-Hand Steadiness,0.94,Abilities,0.134286
1,11-1011.03,Chief Sustainability Officers,Arm-Hand Steadiness,0.50,Abilities,0.071429
2,11-1021.00,General and Operations Managers,Arm-Hand Steadiness,1.25,Abilities,0.178571
3,11-2011.00,Advertising and Promotions Managers,Arm-Hand Steadiness,0.88,Abilities,0.125714
4,11-2021.00,Marketing Managers,Arm-Hand Steadiness,0.88,Abilities,0.125714


In [ ]:
import networkx as nx

G = nx.DiGraph()

# =========================
# 1. Tambahkan occupation
# =========================

for _, row in riasec_profile_normalized.iterrows():
    occupation_id = f"occupation:{row['O*NET-SOC Code']}"

    G.add_node(
        occupation_id,
        type='occupation',
        title=row['Title']
    )

# =========================
# 2. Tambahkan RIASEC
# =========================

for _, row in riasec_profile_normalized.iterrows():

    occupation_id = f"occupation:{row['O*NET-SOC Code']}"

    for interest in riasec:

        interest_id = f"interest:{interest}"

        # Node interest
        G.add_node(
            interest_id,
            type='interest',
            name=interest
        )

        # Occupation → Interest
        G.add_edge(
            occupation_id,
            interest_id,
            relation='has_interest',
            weight=row[interest]
        )

        # Interest → Occupation
        G.add_edge(
            interest_id,
            occupation_id,
            relation='interest_for',
            weight=row[interest]
        )

# =========================
# 3. Tambahkan Essential Skills
# =========================

essential_skill_data = essential_skills_profile.copy()

feature_columns = [
    column for column in essential_skill_data.columns
    if column not in ['O*NET-SOC Code', 'Title', 'Domain']
]

for _, row in essential_skill_data.iterrows():

    occupation_id = f"occupation:{row['O*NET-SOC Code']}"

    for skill in feature_columns:

        if pd.isna(row[skill]):
            continue

        skill_id = f"skill:{skill}"

        # Node skill
        G.add_node(
            skill_id,
            type='skill',
            name=skill
        )

        # Occupation → Skill
        G.add_edge(
            occupation_id,
            skill_id,
            relation='requires_skill',
            weight=row[skill] / 7
        )

        # Skill → Occupation
        G.add_edge(
            skill_id,
            occupation_id,
            relation='required_by',
            weight=row[skill] / 7
        )

print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 939
Jumlah edge : 29276


In [ ]:
# ==========================================
# Menambahkan Competency ke Knowledge Graph
# ==========================================

competency_sources = {
    'Abilities': abilities_profile,
    'Knowledge': knowledge_profile,
    'Essential Skills': essential_skills_profile,
    'Transferable Skills': transferable_skills_profile
}

# Tambahkan node competency
for domain, data in competency_sources.items():

    feature_columns = [
        column for column in data.columns
        if column not in ['O*NET-SOC Code', 'Title', 'Domain']
    ]

    for feature in feature_columns:

        node_id = f"{domain}:{feature}"

        G.add_node(
            node_id,
            type='competency',
            domain=domain,
            name=feature
        )

print("Jumlah node setelah menambahkan competency:",
      G.number_of_nodes())


# ==========================================
# Membangun Personalization Vector
# ==========================================

personalization = {}

# RIASEC
for interest, score in student_riasec.items():

    node_id = f"interest:{interest}"

    if node_id in G:
        personalization[node_id] = score


# Competency
for feature, score in student_competency.items():

    for domain, data in competency_sources.items():

        node_id = f"{domain}:{feature}"

        if node_id in G:
            personalization[node_id] = score


# Normalisasi
total = sum(personalization.values())

personalization = {
    node: score / total
    for node, score in personalization.items()
}


print("\nJumlah node personalization :", len(personalization))

print("\nPersonalization vector:")

for node, score in personalization.items():
    print(f"{node:45} {score:.4f}")

Jumlah node setelah menambahkan competency: 1059

Jumlah node personalization : 22

Personalization vector:
interest:Realistic                            0.0422
interest:Investigative                        0.0584
interest:Artistic                             0.0260
interest:Social                               0.0325
interest:Enterprising                         0.0227
interest:Conventional                         0.0390
Abilities:Deductive Reasoning                 0.0552
Abilities:Inductive Reasoning                 0.0519
Abilities:Mathematical Reasoning              0.0487
Essential Skills:Critical Thinking            0.0584
Transferable Skills:Complex Problem Solving   0.0552
Transferable Skills:Systems Analysis          0.0519
Transferable Skills:Systems Evaluation        0.0487
Transferable Skills:Programming               0.0552
Transferable Skills:Technology Design         0.0519
Abilities:Written Expression                  0.0455
Abilities:Oral Comprehension                

In [ ]:
# ==========================================
# Menghubungkan Occupation dengan Competency
# ==========================================

for domain, data in competency_sources.items():

    feature_columns = [
        column for column in data.columns
        if column not in ['O*NET-SOC Code', 'Title', 'Domain']
    ]

    for _, row in data.iterrows():

        occupation_id = f"occupation:{row['O*NET-SOC Code']}"

        if occupation_id not in G:
            continue

        for feature in feature_columns:

            value = row[feature]

            if pd.isna(value):
                continue

            competency_id = f"{domain}:{feature}"

            # Normalisasi nilai O*NET
            weight = value / 7

            # Occupation → Competency
            G.add_edge(
                occupation_id,
                competency_id,
                relation='requires',
                weight=weight
            )

            # Competency → Occupation
            G.add_edge(
                competency_id,
                occupation_id,
                relation='required_by',
                weight=weight
            )


print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 1059
Jumlah edge : 247676


In [ ]:
# ==========================================
# Personalized PageRank
# ==========================================

pagerank_scores = nx.pagerank(
    G,
    alpha=0.85,
    personalization=personalization,
    weight='weight'
)

# Ambil hanya node occupation
career_scores = []

for node, score in pagerank_scores.items():

    if G.nodes[node].get('type') == 'occupation':

        career_scores.append({
            'O*NET-SOC Code': node.replace('occupation:', ''),
            'Title': G.nodes[node].get('title'),
            'PPR_Score': score
        })


career_ranking = pd.DataFrame(career_scores)

career_ranking = career_ranking.sort_values(
    'PPR_Score',
    ascending=False
).reset_index(drop=True)

career_ranking['Rank'] = career_ranking.index + 1

career_ranking = career_ranking[
    ['Rank', 'O*NET-SOC Code', 'Title', 'PPR_Score']
]

print("Jumlah occupation:", len(career_ranking))

display(career_ranking.head(20))

Jumlah occupation: 923


,Rank,O*NET-SOC Code,Title,PPR_Score
0,1,19-2012.00,Physicists,0.000659
1,2,17-2199.08,Robotics Engineers,0.000653
2,3,17-2112.03,Manufacturing Engineers,0.000647
3,4,17-2031.00,Bioengineers and Biomedical Engineers,0.000640
4,5,29-1211.00,Anesthesiologists,0.000639
5,6,17-2121.00,Marine Engineers and Naval Architects,0.000636
6,7,29-1022.00,Oral and Maxillofacial Surgeons,0.000634
7,8,29-1214.00,Emergency Medicine Physicians,0.000631
8,9,29-1229.03,Urologists,0.000630
9,10,29-1241.00,"Ophthalmologists, Except Pediatric",0.000628


In [ ]:
top_career = career_ranking.iloc[0]

occupation_id = f"occupation:{top_career['O*NET-SOC Code']}"

print("Career :", top_career['Title'])
print("Score  :", top_career['PPR_Score'])

print("\nHubungan dengan profil student:")

for node in G.neighbors(occupation_id):

    if node in personalization:
        print(
            node,
            "→ weight:",
            G[occupation_id][node]['weight']
        )

Career : Physicists
Score  : 0.0006592996990072119

Hubungan dengan profil student:
interest:Realistic → weight: 0.6566666666666667
interest:Investigative → weight: 1.0
interest:Artistic → weight: 0.2916666666666667
interest:Social → weight: 0.18666666666666668
interest:Enterprising → weight: 0.12833333333333333
interest:Conventional → weight: 0.6333333333333333
Abilities:Deductive Reasoning → weight: 0.6871428571428572
Abilities:Inductive Reasoning → weight: 0.6964285714285714
Abilities:Mathematical Reasoning → weight: 0.7678571428571429
Abilities:Oral Comprehension → weight: 0.7235714285714285
Abilities:Written Expression → weight: 0.7142857142857143
Essential Skills:Active Listening → weight: 0.6335714285714287
Essential Skills:Critical Thinking → weight: 0.6871428571428572
Essential Skills:Reading Comprehension → weight: 0.7414285714285713
Essential Skills:Speaking → weight: 0.6699999999999999
Transferable Skills:Complex Problem Solving → weight: 0.6699999999999999
Transferable Ski

In [ ]:
# ==========================================
# Skill Gap Top Career
# ==========================================

top_career = career_ranking.iloc[0]

occupation_code = top_career['O*NET-SOC Code']
occupation_title = top_career['Title']

print("Career :", occupation_title)
print("O*NET-SOC :", occupation_code)


# Ambil kompetensi pekerjaan
occupation_competency = competency_profile_normalized[
    competency_profile_normalized['O*NET-SOC Code'] == occupation_code
].copy()


# Ambil hanya kompetensi yang dimiliki student
student_features = set(student_competency.keys())

occupation_competency = occupation_competency[
    occupation_competency['Feature'].isin(student_features)
].copy()


# Hitung gap
occupation_competency['Student_Level'] = (
    occupation_competency['Feature']
    .map(student_competency)
)

occupation_competency['Gap'] = (
    occupation_competency['Level_Normalized']
    - occupation_competency['Student_Level']
).clip(lower=0)


# Urutkan gap terbesar
skill_gap = occupation_competency.sort_values(
    'Gap',
    ascending=False
).reset_index(drop=True)


skill_gap = skill_gap[
    [
        'Domain',
        'Feature',
        'Level_Normalized',
        'Student_Level',
        'Gap'
    ]
]

print("\nSkill Gap:")
display(skill_gap)

Career : Physicists
O*NET-SOC : 19-2012.00

Skill Gap:


,Domain,Feature,Level_Normalized,Student_Level,Gap
0,Essential Skills,Speaking,0.670000,0.65,0.020000
1,Abilities,Mathematical Reasoning,0.767857,0.75,0.017857
2,Abilities,Written Expression,0.714286,0.70,0.014286
3,Abilities,Inductive Reasoning,0.696429,0.80,0.000000
4,Abilities,Oral Comprehension,0.723571,0.75,0.000000
5,Essential Skills,Active Listening,0.633571,0.70,0.000000
6,Essential Skills,Critical Thinking,0.687143,0.90,0.000000
7,Abilities,Deductive Reasoning,0.687143,0.85,0.000000
8,Essential Skills,Reading Comprehension,0.741429,0.80,0.000000
9,Transferable Skills,Complex Problem Solving,0.670000,0.85,0.000000


In [ ]:
# ==========================================
# Recommendation Result
# ==========================================

top_n = 10

recommendation = career_ranking.head(top_n).copy()

recommendation['PPR_Score'] = recommendation['PPR_Score'].round(6)

print("Career Recommendation:")
display(
    recommendation[
        ['Rank', 'O*NET-SOC Code', 'Title', 'PPR_Score']
    ]
)

Career Recommendation:


,Rank,O*NET-SOC Code,Title,PPR_Score
0,1,19-2012.00,Physicists,0.000659
1,2,17-2199.08,Robotics Engineers,0.000653
2,3,17-2112.03,Manufacturing Engineers,0.000647
3,4,17-2031.00,Bioengineers and Biomedical Engineers,0.000640
4,5,29-1211.00,Anesthesiologists,0.000639
5,6,17-2121.00,Marine Engineers and Naval Architects,0.000636
6,7,29-1022.00,Oral and Maxillofacial Surgeons,0.000634
7,8,29-1214.00,Emergency Medicine Physicians,0.000631
8,9,29-1229.03,Urologists,0.000630
9,10,29-1241.00,"Ophthalmologists, Except Pediatric",0.000628


In [ ]:
# ==========================================
# Detail Career Recommendation
# ==========================================

selected_career = career_ranking.iloc[0]

print("======================================")
print("CAREER RECOMMENDATION")
print("======================================")

print("Rank       :", selected_career['Rank'])
print("Career     :", selected_career['Title'])
print("O*NET-SOC  :", selected_career['O*NET-SOC Code'])
print("PPR Score  :", round(selected_career['PPR_Score'], 6))

print("\nSkill Gap:")

display(
    skill_gap[
        skill_gap['Gap'] > 0
    ][
        ['Domain', 'Feature', 'Gap']
    ]
)

CAREER RECOMMENDATION
Rank       : 1
Career     : Physicists
O*NET-SOC  : 19-2012.00
PPR Score  : 0.000659

Skill Gap:


,Domain,Feature,Gap
0,Essential Skills,Speaking,0.020000
1,Abilities,Mathematical Reasoning,0.017857
2,Abilities,Written Expression,0.014286


In [ ]:
# ==========================================
# Career Detail
# ==========================================

occupation_detail = occupation_data[
    occupation_data['O*NET-SOC Code'] == selected_career['O*NET-SOC Code']
].copy()

display(occupation_detail)

,O*NET-SOC Code,Title,Description
225,19-2012.00,Physicists,"Conduct research into physical phenomena, deve..."


In [ ]:
# ==========================================
# Career Information
# ==========================================

career_info = occupation_detail.iloc[0]

print("======================================")
print("CAREER INFORMATION")
print("======================================")

print("O*NET-SOC Code :", career_info['O*NET-SOC Code'])
print("Career         :", career_info['Title'])
print("Description    :", career_info['Description'])

CAREER INFORMATION
O*NET-SOC Code : 19-2012.00
Career         : Physicists
Description    : Conduct research into physical phenomena, develop theories on the basis of observation and experiments, and devise methods to apply physical laws and theories.


In [ ]:
# ==========================================
# Final Recommendation
# ==========================================

print("======================================")
print("FINAL CAREER RECOMMENDATION")
print("======================================")

print("Rank       :", selected_career['Rank'])
print("Career     :", selected_career['Title'])
print("O*NET-SOC  :", selected_career['O*NET-SOC Code'])
print("PPR Score  :", round(selected_career['PPR_Score'], 6))

print("\nDescription:")
print(career_info['Description'])

print("\nSkill Gap:")

gap_result = skill_gap[
    skill_gap['Gap'] > 0
][
    ['Domain', 'Feature', 'Gap']
].copy()

display(gap_result)

FINAL CAREER RECOMMENDATION
Rank       : 1
Career     : Physicists
O*NET-SOC  : 19-2012.00
PPR Score  : 0.000659

Description:
Conduct research into physical phenomena, develop theories on the basis of observation and experiments, and devise methods to apply physical laws and theories.

Skill Gap:


,Domain,Feature,Gap
0,Essential Skills,Speaking,0.020000
1,Abilities,Mathematical Reasoning,0.017857
2,Abilities,Written Expression,0.014286


In [ ]:
def recommend_career(
    student_riasec,
    student_competency,
    top_n=10
):
    # -----------------------------
    # Personalization
    # -----------------------------

    personalization = {}

    for interest, score in student_riasec.items():

        node_id = f"interest:{interest}"

        if node_id in G:
            personalization[node_id] = score

    for feature, score in student_competency.items():

        for domain in competency_sources.keys():

            node_id = f"{domain}:{feature}"

            if node_id in G:
                personalization[node_id] = score

    total = sum(personalization.values())

    personalization = {
        node: score / total
        for node, score in personalization.items()
    }


    # -----------------------------
    # Personalized PageRank
    # -----------------------------

    pagerank_scores = nx.pagerank(
        G,
        alpha=0.85,
        personalization=personalization,
        weight='weight'
    )


    # -----------------------------
    # Career Ranking
    # -----------------------------

    career_scores = []

    for node, score in pagerank_scores.items():

        if G.nodes[node].get('type') == 'occupation':

            career_scores.append({
                'O*NET-SOC Code': node.replace(
                    'occupation:', ''
                ),
                'Title': G.nodes[node].get('title'),
                'PPR_Score': score
            })

    ranking = pd.DataFrame(career_scores)

    ranking = ranking.sort_values(
        'PPR_Score',
        ascending=False
    ).reset_index(drop=True)

    ranking['Rank'] = ranking.index + 1


    return ranking.head(top_n)

In [ ]:
recommendations = recommend_career(
    student_riasec,
    student_competency
)

display(recommendations)

,O*NET-SOC Code,Title,PPR_Score,Rank
0,19-2012.00,Physicists,0.000659,1
1,17-2199.08,Robotics Engineers,0.000653,2
2,17-2112.03,Manufacturing Engineers,0.000647,3
3,17-2031.00,Bioengineers and Biomedical Engineers,0.000640,4
4,29-1211.00,Anesthesiologists,0.000639,5
5,17-2121.00,Marine Engineers and Naval Architects,0.000636,6
6,29-1022.00,Oral and Maxillofacial Surgeons,0.000634,7
7,29-1214.00,Emergency Medicine Physicians,0.000631,8
8,29-1229.03,Urologists,0.000630,9
9,29-1241.00,"Ophthalmologists, Except Pediatric",0.000628,10


In [ ]:
def get_career_recommendation(
    student_riasec,
    student_competency
):
    # Career ranking
    ranking = recommend_career(
        student_riasec,
        student_competency,
        top_n=10
    )

    # Ambil career terbaik
    selected = ranking.iloc[0]

    occupation_code = selected['O*NET-SOC Code']

    # Career detail
    detail = occupation_data[
        occupation_data['O*NET-SOC Code'] == occupation_code
    ]

    career_info = detail.iloc[0]


    # Ambil competency career
    occupation_competency = competency_profile_normalized[
        competency_profile_normalized['O*NET-SOC Code']
        == occupation_code
    ].copy()

    # Hanya fitur yang dimiliki student
    occupation_competency = occupation_competency[
        occupation_competency['Feature'].isin(
            student_competency.keys()
        )
    ].copy()

    # Student level
    occupation_competency['Student_Level'] = (
        occupation_competency['Feature']
        .map(student_competency)
    )

    # Skill gap
    occupation_competency['Gap'] = (
        occupation_competency['Level_Normalized']
        - occupation_competency['Student_Level']
    ).clip(lower=0)

    skill_gap = occupation_competency[
        occupation_competency['Gap'] > 0
    ].sort_values(
        'Gap',
        ascending=False
    )

    return {
        'ranking': ranking,
        'career': career_info,
        'skill_gap': skill_gap
    }

In [ ]:
result = get_career_recommendation(
    student_riasec,
    student_competency
)

print("Career :", result['career']['Title'])
print("Code   :", result['career']['O*NET-SOC Code'])

print("\nDescription:")
print(result['career']['Description'])

print("\nSkill Gap:")
display(
    result['skill_gap'][
        ['Domain', 'Feature', 'Gap']
    ]
)

Career : Physicists
Code   : 19-2012.00

Description:
Conduct research into physical phenomena, develop theories on the basis of observation and experiments, and devise methods to apply physical laws and theories.

Skill Gap:


,Domain,Feature,Gap
84839,Essential Skills,Speaking,0.020000
19319,Abilities,Mathematical Reasoning,0.017857
46619,Abilities,Written Expression,0.014286


In [ ]:
# ==========================================
# Student Profile 2
# ==========================================

student_riasec_2 = {
    'Realistic': 0.30,
    'Investigative': 0.40,
    'Artistic': 0.90,
    'Social': 0.85,
    'Enterprising': 0.60,
    'Conventional': 0.40
}

student_competency_2 = {
    'Written Expression': 0.85,
    'Oral Comprehension': 0.80,
    'Active Listening': 0.90,
    'Reading Comprehension': 0.80,
    'Speaking': 0.85,
    'Social Perceptiveness': 0.90,
    'Service Orientation': 0.85,
    'Critical Thinking': 0.65,
    'Complex Problem Solving': 0.60,
    'Programming': 0.40,
    'Systems Analysis': 0.40,
    'Systems Evaluation': 0.40,
    'Technology Design': 0.30,
    'Deductive Reasoning': 0.50,
    'Inductive Reasoning': 0.50,
    'Mathematical Reasoning': 0.40
}

In [ ]:
result_2 = get_career_recommendation(
    student_riasec_2,
    student_competency_2
)

print("Career :", result_2['career']['Title'])
print("Code   :", result_2['career']['O*NET-SOC Code'])

print("\nDescription:")
print(result_2['career']['Description'])

print("\nSkill Gap:")
display(
    result_2['skill_gap'][
        ['Domain', 'Feature', 'Gap']
    ]
)

print("\nTop 10:")
display(result_2['ranking'])

Career : Physicists
Code   : 19-2012.00

Description:
Conduct research into physical phenomena, develop theories on the basis of observation and experiments, and devise methods to apply physical laws and theories.

Skill Gap:


,Domain,Feature,Gap
19319,Abilities,Mathematical Reasoning,0.367857
16589,Abilities,Inductive Reasoning,0.196429
3849,Abilities,Deductive Reasoning,0.187143
100309,Transferable Skills,Programming,0.180714
104859,Transferable Skills,Systems Analysis,0.144286
105769,Transferable Skills,Systems Evaluation,0.144286
106679,Transferable Skills,Technology Design,0.128571
86659,Transferable Skills,Complex Problem Solving,0.070000
79379,Essential Skills,Critical Thinking,0.037143



Top 10:


,O*NET-SOC Code,Title,PPR_Score,Rank
0,19-2012.00,Physicists,0.000636,1
1,29-1211.00,Anesthesiologists,0.000636,2
2,29-1214.00,Emergency Medicine Physicians,0.000630,3
3,17-2112.03,Manufacturing Engineers,0.000628,4
4,29-1241.00,"Ophthalmologists, Except Pediatric",0.000627,5
5,29-1229.03,Urologists,0.000626,6
6,17-1011.00,"Architects, Except Landscape and Naval",0.000625,7
7,29-1022.00,Oral and Maxillofacial Surgeons,0.000625,8
8,17-2199.08,Robotics Engineers,0.000622,9
9,17-2121.00,Marine Engineers and Naval Architects,0.000622,10


In [ ]:
# ==========================================
# Cek Pengaruh Personalization
# ==========================================

# Hitung PPR untuk profil kedua
personalization_2 = {}

for interest, score in student_riasec_2.items():

    node_id = f"interest:{interest}"

    if node_id in G:
        personalization_2[node_id] = score


for feature, score in student_competency_2.items():

    for domain in competency_sources.keys():

        node_id = f"{domain}:{feature}"

        if node_id in G:
            personalization_2[node_id] = score


total = sum(personalization_2.values())

personalization_2 = {
    node: score / total
    for node, score in personalization_2.items()
}


# PPR
pagerank_2 = nx.pagerank(
    G,
    alpha=0.85,
    personalization=personalization_2,
    weight='weight'
)


# Ambil beberapa career
check_careers = [
    'Physicists',
    'Robotics Engineers',
    'Anesthesiologists',
    'Architects, Except Landscape and Naval'
]

check_result = []

for node, score in pagerank_2.items():

    if G.nodes[node].get('type') == 'occupation':

        title = G.nodes[node].get('title')

        if title in check_careers:

            check_result.append({
                'Career': title,
                'PPR_Score': score
            })


check_result = pd.DataFrame(check_result)

display(
    check_result.sort_values(
        'PPR_Score',
        ascending=False
    )
)

,Career,PPR_Score
2,Physicists,0.000636
3,Anesthesiologists,0.000636
0,"Architects, Except Landscape and Naval",0.000625
1,Robotics Engineers,0.000622


In [ ]:
# ==========================================
# Test Multiple Student Profiles
# ==========================================

profiles = {
    'Investigative Technical': {
        'riasec': {
            'Realistic': 0.70,
            'Investigative': 0.95,
            'Artistic': 0.20,
            'Social': 0.30,
            'Enterprising': 0.20,
            'Conventional': 0.60
        },
        'competency': {
            'Mathematical Reasoning': 0.90,
            'Deductive Reasoning': 0.90,
            'Inductive Reasoning': 0.90,
            'Critical Thinking': 0.90,
            'Complex Problem Solving': 0.90,
            'Programming': 0.85,
            'Systems Analysis': 0.85,
            'Systems Evaluation': 0.85,
            'Technology Design': 0.80
        }
    },

    'Artistic Social': {
        'riasec': {
            'Realistic': 0.20,
            'Investigative': 0.30,
            'Artistic': 0.95,
            'Social': 0.90,
            'Enterprising': 0.60,
            'Conventional': 0.30
        },
        'competency': {
            'Written Expression': 0.90,
            'Speaking': 0.90,
            'Active Listening': 0.90,
            'Reading Comprehension': 0.85,
            'Social Perceptiveness': 0.90,
            'Service Orientation': 0.90,
            'Critical Thinking': 0.70
        }
    },

    'Enterprising Social': {
        'riasec': {
            'Realistic': 0.30,
            'Investigative': 0.30,
            'Artistic': 0.40,
            'Social': 0.85,
            'Enterprising': 0.95,
            'Conventional': 0.70
        },
        'competency': {
            'Speaking': 0.90,
            'Oral Comprehension': 0.90,
            'Active Listening': 0.85,
            'Social Perceptiveness': 0.90,
            'Service Orientation': 0.85,
            'Written Expression': 0.80,
            'Critical Thinking': 0.75
        }
    },

    'Realistic Technical': {
        'riasec': {
            'Realistic': 0.95,
            'Investigative': 0.70,
            'Artistic': 0.20,
            'Social': 0.30,
            'Enterprising': 0.40,
            'Conventional': 0.50
        },
        'competency': {
            'Mathematical Reasoning': 0.85,
            'Deductive Reasoning': 0.85,
            'Inductive Reasoning': 0.80,
            'Complex Problem Solving': 0.85,
            'Technology Design': 0.90,
            'Systems Analysis': 0.80,
            'Systems Evaluation': 0.80,
            'Programming': 0.70
        }
    }
}


# Jalankan semua profil
test_results = []

for profile_name, profile in profiles.items():

    result = recommend_career(
        profile['riasec'],
        profile['competency'],
        top_n=5
    )

    for _, row in result.iterrows():

        test_results.append({
            'Profile': profile_name,
            'Rank': row['Rank'],
            'Career': row['Title'],
            'PPR_Score': row['PPR_Score']
        })


test_results = pd.DataFrame(test_results)

display(test_results)

,Profile,Rank,Career,PPR_Score
0,Investigative Technical,1,Physicists,0.000680
1,Investigative Technical,2,Robotics Engineers,0.000678
2,Investigative Technical,3,Manufacturing Engineers,0.000663
3,Investigative Technical,4,Bioengineers and Biomedical Engineers,0.000661
4,Investigative Technical,5,Marine Engineers and Naval Architects,0.000651
5,Artistic Social,1,"Architecture Teachers, Postsecondary",0.000640
6,Artistic Social,2,Anesthesiologists,0.000635
7,Artistic Social,3,"Ophthalmologists, Except Pediatric",0.000631
8,Artistic Social,4,Emergency Medicine Physicians,0.000631
9,Artistic Social,5,"Education Administrators, Kindergarten through...",0.000630


In [ ]:
# ==========================================
# Baseline Similarity
# ==========================================

def similarity_recommendation(
    student_riasec,
    student_competency,
    top_n=10
):
    results = []

    for _, occupation in riasec_profile_normalized.iterrows():

        # -----------------------------
        # RIASEC Fit
        # -----------------------------

        interest_differences = []

        for feature in riasec:

            student_value = student_riasec.get(
                feature,
                0
            )

            occupation_value = occupation[feature]

            interest_differences.append(
                abs(student_value - occupation_value)
            )

        interest_fit = (
            1 - sum(interest_differences)
            / len(interest_differences)
        )


        # -----------------------------
        # Competency Fit
        # -----------------------------

        occupation_code = occupation[
            'O*NET-SOC Code'
        ]

        occupation_competency = competency_profile_normalized[
            competency_profile_normalized[
                'O*NET-SOC Code'
            ] == occupation_code
        ]

        competency_differences = []

        for feature, student_value in student_competency.items():

            matching = occupation_competency[
                occupation_competency['Feature']
                == feature
            ]

            if len(matching) == 0:
                continue

            occupation_value = matching[
                'Level_Normalized'
            ].iloc[0]

            competency_differences.append(
                abs(student_value - occupation_value)
            )

        if len(competency_differences) > 0:

            competency_fit = (
                1 - sum(competency_differences)
                / len(competency_differences)
            )

        else:
            competency_fit = 0


        # -----------------------------
        # Overall Similarity
        # -----------------------------

        overall_score = (
            0.5 * interest_fit
            + 0.5 * competency_fit
        )

        results.append({
            'O*NET-SOC Code': occupation_code,
            'Title': occupation['Title'],
            'Similarity_Score': overall_score
        })


    result = pd.DataFrame(results)

    result = result.sort_values(
        'Similarity_Score',
        ascending=False
    ).reset_index(drop=True)

    result['Rank'] = result.index + 1

    return result.head(top_n)

In [ ]:
baseline_result = similarity_recommendation(
    student_riasec,
    student_competency,
    top_n=10
)

display(baseline_result)

,O*NET-SOC Code,Title,Similarity_Score,Rank
0,19-2012.00,Physicists,0.868160,1
1,17-2112.01,Human Factors Engineers and Ergonomists,0.846567,2
2,19-2021.00,Atmospheric and Space Scientists,0.839903,3
3,19-1029.02,Molecular and Cellular Biologists,0.839420,4
4,29-1224.00,Radiologists,0.838614,5
5,15-1221.00,Computer and Information Research Scientists,0.837041,6
6,19-2011.00,Astronomers,0.836513,7
7,17-2161.00,Nuclear Engineers,0.836104,8
8,17-2031.00,Bioengineers and Biomedical Engineers,0.835300,9
9,29-1222.00,"Physicians, Pathologists",0.833663,10


In [ ]:
# ==========================================
# PPR vs Similarity
# ==========================================

ppr_result = recommend_career(
    student_riasec,
    student_competency,
    top_n=10
)

comparison = pd.DataFrame({
    'PPR Rank': ppr_result['Title'].tolist(),
    'Similarity Rank': baseline_result['Title'].tolist()
})

display(comparison)

,PPR Rank,Similarity Rank
0,Physicists,Physicists
1,Robotics Engineers,Human Factors Engineers and Ergonomists
2,Manufacturing Engineers,Atmospheric and Space Scientists
3,Bioengineers and Biomedical Engineers,Molecular and Cellular Biologists
4,Anesthesiologists,Radiologists
5,Marine Engineers and Naval Architects,Computer and Information Research Scientists
6,Oral and Maxillofacial Surgeons,Astronomers
7,Emergency Medicine Physicians,Nuclear Engineers
8,Urologists,Bioengineers and Biomedical Engineers
9,"Ophthalmologists, Except Pediatric","Physicians, Pathologists"


In [ ]:
# ==========================================
# Top 10 Overlap
# ==========================================

ppr_top10 = set(ppr_result['Title'])
similarity_top10 = set(baseline_result['Title'])

overlap = ppr_top10.intersection(
    similarity_top10
)

print("PPR Top 10        :", len(ppr_top10))
print("Similarity Top 10 :", len(similarity_top10))
print("Career yang sama  :", len(overlap))

print("\nCareer yang sama:")
for career in overlap:
    print("-", career)

PPR Top 10        : 10
Similarity Top 10 : 10
Career yang sama  : 2

Career yang sama:
- Physicists
- Bioengineers and Biomedical Engineers
